In [13]:
from dotenv import load_dotenv
load_dotenv()

import json

from rag_helper import RAGBase

from openai import OpenAI
openai_client = OpenAI()

import time
from sqlitesearch import TextSearchIndex

In [4]:
# The developer prompt

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [7]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False


In [9]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_JQX4GL47AWmRKea43voVZB6o', name='search', type='function_call', id='fc_07e891b12743b6fc006a2f2ee9b6c08191a78f9a5617becdbb', namespace=None, status='completed')]

[
    ResponseFunctionToolCall(
    arguments='{
        "query":"join course discovered course can I join enrollment late join FAQ"
        }', 
    call_id='call_JQX4GL47AWmRKea43voVZB6o',
    name='search',
    type='function_call',
    id='fc_07e891b12743b6fc006a2f2ee9b6c08191a78f9a5617becdbb', 
    namespace=None, 
    status='completed'
    )
]

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [14]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [15]:

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}


In [ ]:
# The full agent loop

In [17]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, though, you’ll need to submit your project while submissions are still open.

Would you like to explore anything else about the course, like certificates or how to follow it self-paced?


In [18]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally install local Ollama model run locally"}
iteration #2...
function_call: search {"query":"Ollama localhost 11434 ollama run llama3 Python client ollama chat"}
iteration #3...
ASSISTANT:
To run **Ollama locally**, first install it for your operating system:

- **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
- **Windows**: download the `.msi`
- **Linux**: run:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then start a model locally:

```bash
ollama run llama3
```

That will download the model, start it locally, and open a chat-like prompt.

You can also check that the local server is running with:

```bash
curl http://localhost:11434
```

If you want to use it from Python:

```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": "Hello!"}]
)

print(response['message']['content'])
`

'To run **Ollama locally**, first install it for your operating system:\n\n- **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n- **Windows**: download the `.msi`\n- **Linux**: run:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nThen start a model locally:\n\n```bash\nollama run llama3\n```\n\nThat will download the model, start it locally, and open a chat-like prompt.\n\nYou can also check that the local server is running with:\n\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python:\n\n```bash\npip install ollama\n```\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nIf you’re using Ollama in the course’s agent work, note that `llama3.1` and similar models support tool/function calling via `ollama.chat(..., tools=[...])`.\n\nWould you like help with installing it 

In [20]:
agent_loop(instructions, "do you know how to program a forecasting tool for pharma sector using c++?")

iteration #1...
function_call: search {"query":"C++ forecasting tool pharma sector time series forecasting pharma FAQ"}
function_call: search {"query":"pharma forecasting tool programming C++ demand forecasting FAQ"}
iteration #2...
ASSISTANT:
Yes — you can build a forecasting tool for the pharma sector in C++, but the best design depends on what you want to forecast:

- drug demand / sales
- inventory needs
- patient volume
- production planning
- revenue forecasting

A typical approach in C++ would be:

1. **Collect and clean data**
   - historical sales
   - seasonality
   - market events
   - promotions
   - regulatory changes
   - disease/incidence trends

2. **Choose a forecasting method**
   - simple moving average
   - exponential smoothing
   - ARIMA / SARIMA
   - regression with external features
   - machine learning models if you integrate a library

3. **Implement the model in C++**
   - use STL for data handling
   - use Eigen / Armadillo for matrix operations
   - use a 

'Yes — you can build a forecasting tool for the pharma sector in C++, but the best design depends on what you want to forecast:\n\n- drug demand / sales\n- inventory needs\n- patient volume\n- production planning\n- revenue forecasting\n\nA typical approach in C++ would be:\n\n1. **Collect and clean data**\n   - historical sales\n   - seasonality\n   - market events\n   - promotions\n   - regulatory changes\n   - disease/incidence trends\n\n2. **Choose a forecasting method**\n   - simple moving average\n   - exponential smoothing\n   - ARIMA / SARIMA\n   - regression with external features\n   - machine learning models if you integrate a library\n\n3. **Implement the model in C++**\n   - use STL for data handling\n   - use Eigen / Armadillo for matrix operations\n   - use a stats or ML library if needed\n\n4. **Evaluate accuracy**\n   - MAE\n   - RMSE\n   - MAPE\n   - backtesting on historical data\n\n5. **Build the tool**\n   - command-line app, desktop app, or service/API\n   - outpu

In [21]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions.

If you want, I can also help with questions about certificates, deadlines, or how to follow the course self-paced.


'Yes — you can still join the course.\n\nOne important note: if you want a certificate, you need to submit your project while the course is still accepting submissions.\n\nIf you want, I can also help with questions about certificates, deadlines, or how to follow the course self-paced.'

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late enrollment can I join discovered the course course FAQ"}
iteration #2...
function_call: search {"query":"self-paced mode certificate live cohort project submit while accepting submissions peer-review course FAQ"}
iteration #3...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only awarded for the live cohort, not self-paced participation.

If you want, I can also explain how certification and project submission work.


'Yes, you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only awarded for the live cohort, not self-paced participation.\n\nIf you want, I can also explain how certification and project submission work.'

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit what is queen gambit chess opening"}
iteration #2...
function_call: search {"query":"queen gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it doesn’t look like a course-related question.

If you meant a chess term, “Queen’s Gambit” is a chess opening, but I can’t go beyond the FAQ database here. If you have a course-related question, send it over and I’ll help. Are there other areas that you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so it doesn’t look like a course-related question.\n\nIf you meant a chess term, “Queen’s Gambit” is a chess opening, but I can’t go beyond the FAQ database here. If you have a course-related question, send it over and I’ll help. Are there other areas that you want to explore?'